# Ill-conditioning: the same equation, in the wrong units

**Book:** §5.7, Figure 5.6 &nbsp;·&nbsp; `ch05/ill_conditioning.ipynb`

A slab with volumetric heating, written the way an engineer writes it — **in SI units**:

$$k\,\frac{d^2T}{dx^2} + q = 0,\qquad T(0)=T(L)=300\ \text{K},$$

with $L = 0.05$ m, $k = 15$ W/m·K, $q = 10^6$ W/m³. Exact: $T = 300 + \frac{q}{2k}x(L-x)$, a peak rise
of only **20.8 K**.

Look at the magnitudes. The input $x \in [0, 0.05]$. The output $T \approx 300$. And
$T'' = -q/k = -66{,}667$. **Nothing in this problem is $O(1)$**, and the network is initialised
assuming everything is.

**The remedies, all four implemented below:**
1. **Non-dimensionalise** — $x^*=x/L$, $\theta=(T-T_w)/\Delta T$ with $\Delta T = qL^2/2k$, giving
   $\theta'' + 2 = 0$, $\theta(0)=\theta(1)=0$. *Identical physics. Every term now $O(1)$.*
2. **Anneal / hard BCs** — $\theta = x(1-x)\mathcal N$, so the loss is a pure residual.
3. **Polish with L-BFGS** after Adam (freeze the collocation points first — L-BFGS needs a closure
   and dislikes a fresh batch each step).
4. **Report seed sensitivity** — five seeds, and quote the spread. *A result that moves across
   seeds is not a result.*

In [ ]:
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
def mlp(s, seed=0):
    torch.manual_seed(seed); L = []
    for i in range(len(s)-1):
        L.append(nn.Linear(s[i], s[i+1]))
        if i < len(s)-2: L.append(nn.Tanh())
    return nn.Sequential(*L)
rel = lambda p, e: float(np.sqrt(np.mean((p-e)**2)/np.mean((e-e.mean())**2 + 1e-30)))
M = {}

In [ ]:
Lm, kc, qv, Tw = 0.05, 15.0, 1.0e6, 300.0
Texact = lambda x: Tw + (qv/(2*kc))*x*(Lm - x)
xg = np.linspace(0, Lm, 400); Tex = Texact(xg)
DT = qv*Lm**2/(2*kc)                      # 41.67 K, the natural temperature scale
print(f'[5.7] peak rise = {Tex.max()-Tw:.2f} K,  T\'\' = {-qv/kc:.0f} K/m^2')

# --- (a) the naive, fully dimensional PINN: soft BCs, raw x, raw T ---
net = mlp([1,32,32,1], 0); opt = torch.optim.Adam(net.parameters(), 2e-3)
for e in range(6000):
    opt.zero_grad()
    x = (torch.rand(512,1)*Lm).requires_grad_(True)
    T = net(x)
    res = kc*g1(g1(T,x),x) + qv                       # O(1e6). The BC term is O(1e5).
    z = torch.zeros(1,1); o = torch.full((1,1), Lm)
    ((res**2).mean() + ((net(z)-Tw)**2 + (net(o)-Tw)**2).sum()).backward(); opt.step()
xt = torch.tensor(xg, dtype=torch.float32).reshape(-1,1)
with torch.no_grad(): Tdim = net(xt).numpy().ravel()
M['ill_dim'] = rel(Tdim, Tex)
print(f'[5.7] dimensional PINN : rel L2 = {M["ill_dim"]:.3f}')

# --- (b) non-dimensionalised: x* = x/L, theta = (T-Tw)/DT  ->  theta'' + 2 = 0 ---
def nondim(seed, lbfgs=False):
    net = mlp([1,32,32,1], seed)
    th = lambda s: s*(1-s)*net(s)                     # theta(0)=theta(1)=0, hard
    opt = torch.optim.Adam(net.parameters(), 3e-3)
    for e in range(3000):
        opt.zero_grad()
        s = torch.rand(512,1).requires_grad_(True)
        ((g1(g1(th(s),s),s) + 2.0)**2).mean().backward(); opt.step()   # every term O(1)
    if lbfgs:                                          # polish: freeze the points, then L-BFGS
        s = torch.rand(512,1).requires_grad_(True)
        lb = torch.optim.LBFGS(net.parameters(), max_iter=300, history_size=50,
                               tolerance_grad=1e-12, tolerance_change=1e-14,
                               line_search_fn='strong_wolfe')
        def closure():
            lb.zero_grad()
            l = ((g1(g1(th(s),s),s) + 2.0)**2).mean()
            l.backward(); return l
        lb.step(closure)
    st = torch.tensor(xg/Lm, dtype=torch.float32).reshape(-1,1)
    with torch.no_grad(): Tp = Tw + DT*th(st).numpy().ravel()
    return Tp, rel(Tp, Tex)

Tnd, M['ill_nd'] = nondim(0)
Tlb, M['ill_lbfgs'] = nondim(0, lbfgs=True)
seeds = [nondim(s)[1] for s in range(5)]
M['ill_seed_mean'] = float(np.mean(seeds)); M['ill_seed_std'] = float(np.std(seeds))
M['ill_seeds'] = [float(s) for s in seeds]
print(f'[5.7] non-dimensional : rel L2 = {M["ill_nd"]:.2e}')
print(f'[5.7] + L-BFGS polish : rel L2 = {M["ill_lbfgs"]:.2e}')
print(f'[5.7] seed spread (5) : {M["ill_seed_mean"]:.2e} +/- {M["ill_seed_std"]:.1e}')

fig, ax = plt.subplots(1, 3, figsize=(14.5, 4.2))
ax[0].plot(xg*1e3, Tex, 'g', lw=2.8, alpha=.6, label='exact')
ax[0].plot(xg*1e3, Tdim, 'r--', lw=1.8, label=f'PINN in SI units (rel $L_2$={M["ill_dim"]:.2f})')
ax[0].set_xlabel('x (mm)'); ax[0].set_ylabel('T (K)'); ax[0].grid(alpha=.3); ax[0].legend(fontsize=9)
ax[0].set_title('(a) As written: $k\\,T\'\'+q=0$, SI units\nresidual $O(10^6)$, $T\\sim300$', fontsize=10.5)
ax[1].plot(xg*1e3, Tex, 'g', lw=2.8, alpha=.6, label='exact')
ax[1].plot(xg*1e3, Tnd, 'r--', lw=1.8, label=f'non-dimensional (rel $L_2$={M["ill_nd"]:.0e})')
ax[1].set_xlabel('x (mm)'); ax[1].set_ylabel('T (K)'); ax[1].grid(alpha=.3); ax[1].legend(fontsize=9)
ax[1].set_title("(b) Same equation, rescaled: $\\theta''+2=0$\nidentical physics, $O(1)$ throughout",
                fontsize=10.5)
names = ['SI units', 'non-dim.', 'non-dim.\n+ L-BFGS']
vals = [M['ill_dim'], M['ill_nd'], M['ill_lbfgs']]
b = ax[2].bar(names, vals, color=['tab:red','tab:blue','tab:green'], alpha=.85)
ax[2].errorbar([1], [M['ill_seed_mean']], yerr=[M['ill_seed_std']], fmt='k_', ms=18, capsize=6,
               lw=1.5, label='spread over 5 seeds')
ax[2].set_yscale('log'); ax[2].set_ylabel('relative $L_2$ error'); ax[2].grid(alpha=.3, axis='y')
ax[2].legend(fontsize=8); ax[2].set_title('(c) One division buys four orders\nof magnitude', fontsize=10.5)
for r_, v_ in zip(b, vals):
    ax[2].text(r_.get_x()+r_.get_width()/2, v_*1.35, f'{v_:.1e}', ha='center', fontsize=8.5)
plt.tight_layout(); plt.show()